In [2]:
import os
import pickle
import h5py
import numpy as np
import pandas as pd
from pathlib import Path
from typing import List, Dict, Any, Optional


class CaeteOutputRun:
    """
    Classe para representar uma pasta de execução (run) do CAETE.
    Gerencia a estrutura de arquivos e pastas de uma simulação.
    """
    
    def __init__(self, run_path: str):
        """
        Inicializa um objeto CaeteOutputRun a partir do caminho da pasta.
        
        Args:
            run_path: Caminho para a pasta da run
        """
        self.run_path = Path(run_path)
        self.run_name = self.run_path.name
        
        # Inicializa atributos
        self.gridcells: List[str] = []
        self.gridcell_spins: Dict[str, int] = {}
        self.gridcell_vars: Dict[str, List[str]] = {}
        self.gridcell_years: Dict[str, tuple] = {}
        
        # Verifica arquivos obrigatórios
        self.caete_h5 = self.run_path / "CAETE.h5"
        self.state_start = self.run_path / "CAETE_STATE_START.pkz"
        self.state_end = self.run_path / "CAETE_STATE_END.pkz"
        self.nc_outputs_dir = self.run_path / "nc_outputs"
        
        # Processa a estrutura
        self._process_structure()
    
    def _process_structure(self):
        """Processa a estrutura da pasta da run."""
        # Encontra todas as pastas gridcell
        for item in self.run_path.iterdir():
            if item.is_dir() and item.name.startswith("gridcell"):
                gridcell_name = item.name
                self.gridcells.append(gridcell_name)
                
                # Conta arquivos spin*.pkz
                spin_files = list(item.glob("spin*.pkz"))
                self.gridcell_spins[gridcell_name] = len(spin_files)
                
                # Para cada gridcell, processa os arquivos spin para obter variáveis e anos
                if spin_files:
                    # Lê o primeiro arquivo spin para obter variáveis e anos
                    first_spin = spin_files[0]
                    try:
                        with open(first_spin, 'rb') as f:
                            data = pickle.load(f)
                        
                        # Extrai variáveis (assumindo que data é um dicionário)
                        if isinstance(data, dict):
                            vars_list = list(data.keys())
                            self.gridcell_vars[gridcell_name] = vars_list
                            
                            # Tenta extrair range de anos se disponível
                            # Assumindo que pode haver uma variável 'year' ou similar
                            if 'year' in data:
                                years = data['year']
                                if isinstance(years, (list, np.ndarray)):
                                    self.gridcell_years[gridcell_name] = (min(years), max(years))
                    except Exception as e:
                        print(f"Erro ao ler {first_spin}: {e}")
                        self.gridcell_vars[gridcell_name] = []
                        self.gridcell_years[gridcell_name] = (None, None)
    
    def get_start_state(self) -> Optional[pd.DataFrame]:
        """
        Lê o arquivo CAETE_STATE_START.pkz e retorna como DataFrame.
        
        Returns:
            DataFrame com os dados do estado inicial ou None se não encontrar
        """
        if not self.state_start.exists():
            print(f"Arquivo não encontrado: {self.state_start}")
            return None
        
        try:
            with open(self.state_start, 'rb') as f:
                data = pickle.load(f)
            
            # Converte para DataFrame se for dicionário
            if isinstance(data, dict):
                return pd.DataFrame(data)
            else:
                return pd.DataFrame([data]) if data is not None else None
        except Exception as e:
            print(f"Erro ao ler {self.state_start}: {e}")
            return None
    
    def get_end_state(self) -> Optional[pd.DataFrame]:
        """
        Lê o arquivo CAETE_STATE_END.pkz e retorna como DataFrame.
        
        Returns:
            DataFrame com os dados do estado final ou None se não encontrar
        """
        if not self.state_end.exists():
            print(f"Arquivo não encontrado: {self.state_end}")
            return None
        
        try:
            with open(self.state_end, 'rb') as f:
                data = pickle.load(f)
            
            # Converte para DataFrame se for dicionário
            if isinstance(data, dict):
                return pd.DataFrame(data)
            else:
                return pd.DataFrame([data]) if data is not None else None
        except Exception as e:
            print(f"Erro ao ler {self.state_end}: {e}")
            return None
    
    def summary_nc(self) -> Dict[str, Any]:
        """
        Sumariza os arquivos netCDF na pasta nc_outputs.
        
        Returns:
            Dicionário com informações sobre os arquivos .nc
        """
        nc_files = list(self.nc_outputs_dir.glob("*.nc")) if self.nc_outputs_dir.exists() else []
        
        nc_info = {
            'count': len(nc_files),
            'files': []
        }
        
        for nc_file in nc_files:
            file_size = nc_file.stat().st_size
            nc_info['files'].append({
                'name': nc_file.name,
                'size_bytes': file_size,
                'size_mb': file_size / (1024 * 1024)
            })
        
        return nc_info
    
    def summary(self) -> Dict[str, Any]:
        """
        Gera um sumário completo de todos os componentes da run.
        
        Returns:
            Dicionário com todas as informações da run
        """
        nc_summary = self.summary_nc()
        
        summary_dict = {
            'run_name': self.run_name,
            'run_path': str(self.run_path),
            'gridcells': {
                'count': len(self.gridcells),
                'names': self.gridcells,
                'spin_counts': self.gridcell_spins,
                'variables': self.gridcell_vars,
                'year_ranges': self.gridcell_years
            },
            'state_files': {
                'has_start': self.state_start.exists(),
                'has_end': self.state_end.exists()
            },
            'nc_outputs': nc_summary,
            'caete_h5_exists': self.caete_h5.exists()
        }
        
        return summary_dict
    
    def __repr__(self):
        return f"CaeteOutputRun(run_name='{self.run_name}', gridcells={len(self.gridcells)})"


class CaeteOutputs:
    """
    Classe principal para gerenciar múltiplas execuções do CAETE.
    A pasta principal deve conter apenas subpastas (runs).
    """
    
    def __init__(self, path: str):
        """
        Inicializa o objeto CaeteOutputs verificando a estrutura da pasta.
        
        Args:
            path: Caminho para a pasta principal dos outputs
        
        Raises:
            ValueError: Se a pasta contiver arquivos soltos ou estrutura inválida
        """
        self.path = Path(path)
        
        if not self.path.exists():
            raise ValueError(f"O caminho {path} não existe")
        
        if not self.path.is_dir():
            raise ValueError(f"O caminho {path} não é um diretório")
        
        # Lista de nomes das runs (subpastas)
        self.run_names: List[str] = []
        # Dicionário de objetos CaeteOutputRun
        self.runs: Dict[str, CaeteOutputRun] = {}
        
        # Processa a estrutura
        self._process_structure()
    
    def _process_structure(self):
        """Processa a estrutura da pasta principal."""
        for item in self.path.iterdir():
            if item.is_dir():
                # É uma subpasta - consideramos como uma run
                self.run_names.append(item.name)
                self.runs[item.name] = CaeteOutputRun(str(item))
            else:
                # Encontrou um arquivo solto
                raise ValueError(
                    f"A pasta {self.path} contém arquivos soltos. "
                    f"Apenas subpastas são permitidas. Arquivo encontrado: {item.name}"
                )
        
        if not self.run_names:
            raise ValueError(f"Nenhuma subpasta (run) encontrada em {self.path}")
    
    def get_run(self, run_name: str) -> Optional[CaeteOutputRun]:
        """
        Retorna um objeto CaeteOutputRun específico.
        
        Args:
            run_name: Nome da run (subpasta)
        
        Returns:
            Objeto CaeteOutputRun ou None se não existir
        """
        return self.runs.get(run_name)
    
    def summary(self) -> Dict[str, Any]:
        """
        Gera um sumário de todas as runs disponíveis.
        
        Returns:
            Dicionário com as informações básicas das runs
        """
        runs_summary = []
        
        for run_name, run_obj in self.runs.items():
            run_summary = {
                'run_name': run_name,
                'gridcells_count': len(run_obj.gridcells),
                'total_spins': sum(run_obj.gridcell_spins.values()),
                'has_nc_outputs': run_obj.nc_outputs_dir.exists(),
                'has_state_files': run_obj.state_start.exists() and run_obj.state_end.exists()
            }
            runs_summary.append(run_summary)
        
        return {
            'main_path': str(self.path),
            'total_runs': len(self.run_names),
            'runs': runs_summary
        }
    
    def detailed_summary(self) -> Dict[str, Any]:
        """
        Gera um sumário detalhado de todas as runs.
        
        Returns:
            Dicionário com os sumários detalhados de cada run
        """
        detailed = {
            'main_path': str(self.path),
            'total_runs': len(self.run_names),
            'runs': {}
        }
        
        for run_name, run_obj in self.runs.items():
            detailed['runs'][run_name] = run_obj.summary()
        
        return detailed
    
    def __repr__(self):
        return f"CaeteOutputs(path='{self.path}', runs={self.run_names})"
    
    def __len__(self):
        return len(self.run_names)
    
    def __iter__(self):
        return iter(self.runs.values())
    
    def __getitem__(self, key):
        return self.runs[key]


# Exemplo de uso:
if __name__ == "__main__":
    # Exemplo de como usar as classes
    try:
        # Para usar, forneça o caminho da pasta principal
        caete_outputs = CaeteOutputs("./outputs")
        
        # Sumário básico
        print("=== Sumário Básico ===")
        print(caete_outputs.summary())
        
        # Acessar uma run específica
        primeira_run = caete_outputs.run_names[0]
        run_obj = caete_outputs.get_run(primeira_run)
        
        if run_obj:
            print(f"\n=== Detalhes da Run: {primeira_run} ===")
            print(run_obj.summary())
            
            # Ler estados
            start_state = run_obj.get_start_state()
            if start_state is not None:
                print(f"\nEstado inicial carregado: {start_state.shape}")
            
            # Sumário dos arquivos nc
            print(f"\nSumário NC: {run_obj.summary_nc()}")
    
    except ValueError as e:
        print(f"Erro: {e}")

Erro ao ler outputs/lu3003/gridcell186-239/spin01.pkz: invalid load key, 'x'.
Erro ao ler outputs/lu2/gridcell186-239/spin01.pkz: invalid load key, 'x'.
=== Sumário Básico ===
{'main_path': 'outputs', 'total_runs': 2, 'runs': [{'run_name': 'lu3003', 'gridcells_count': 1, 'total_spins': 35, 'has_nc_outputs': False, 'has_state_files': False}, {'run_name': 'lu2', 'gridcells_count': 1, 'total_spins': 35, 'has_nc_outputs': True, 'has_state_files': False}]}

=== Detalhes da Run: lu3003 ===
{'run_name': 'lu3003', 'run_path': 'outputs/lu3003', 'gridcells': {'count': 1, 'names': ['gridcell186-239'], 'spin_counts': {'gridcell186-239': 35}, 'variables': {'gridcell186-239': []}, 'year_ranges': {'gridcell186-239': (None, None)}}, 'state_files': {'has_start': False, 'has_end': False}, 'nc_outputs': {'count': 0, 'files': []}, 'caete_h5_exists': True}
Arquivo não encontrado: outputs/lu3003/CAETE_STATE_START.pkz

Sumário NC: {'count': 0, 'files': []}
